In [2]:
!pip install -q transformers datasets peft

import pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice
train = pd.read_csv('C:\\Users\\SAGAR\\OneDrive\\Desktop\\dl_genai\\dl-genai-project\\data\\train.csv')   # <-- apna path
OPTS = ['A','B','C','D','E']
tok = AutoTokenizer.from_pretrained("bert-base-uncased")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
label_map = {'A':0,'B':1,'C':2,'D':3,'E':4}
train['label'] = train['answer'].map(label_map)
print("ANSWER Q1:", train.iloc[150]['label'])

ANSWER Q1: 2


In [4]:
row0 = train.iloc[0]
formatted = str(row0['prompt']) + " [SEP] " + str(row0['B'])
print("ANSWER Q2:", len(formatted))

ANSWER Q2: 407


In [5]:
inputs_5 = [str(row0['prompt']) + " [SEP] " + str(row0[o]) for o in OPTS]
enc = tok(inputs_5, padding="max_length", truncation=True,
          max_length=128, return_tensors="pt")
input_ids = enc["input_ids"].unsqueeze(0)     # reshape -> (1, 5, 128)
print("shape:", input_ids.shape)
print("ANSWER Q3:", input_ids.shape[1])       # second dimension = 5

shape: torch.Size([1, 5, 128])
ANSWER Q3: 5


In [6]:
all_texts = []
for i in range(16):
    r = train.iloc[i]
    all_texts += [str(r['prompt']) + " [SEP] " + str(r[o]) for o in OPTS]

enc16 = tok(all_texts, padding="max_length", truncation=True,
            max_length=128, return_tensors="pt")
ids16 = enc16["input_ids"].view(16, 5, 128)
print("shape:", ids16.shape)
print("ANSWER Q4:", ids16.numel())            # 16*5*128 = 10240

shape: torch.Size([16, 5, 128])
ANSWER Q4: 10240


In [7]:
mc_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
# (LOAD REPORT warning normal hai - naya MC head init hota hai)

batch = {k: v.unsqueeze(0) for k, v in enc.items()}     # (1, 5, 128) from Q3
with torch.no_grad():
    out = mc_model(**batch)
print("logits shape:", out.logits.shape)                 # [1, 5]
print("ANSWER Q5:", out.logits.shape[1])                 # 5 logits

label0 = torch.tensor([int(train.iloc[0]['label'])])
with torch.no_grad():
    out_l = mc_model(**batch, labels=label0)
print("loss:", out_l.loss, "| ANSWER Q6:", out_l.loss.dim())   # scalar -> 0 dimensions

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


logits shape: torch.Size([1, 5])
ANSWER Q5: 5
loss: tensor(1.6128) | ANSWER Q6: 0


In [8]:
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["query","value"],
                      lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_CLS)
lora_model = get_peft_model(AutoModelForMultipleChoice.from_pretrained("bert-base-uncased"),
                            lora_cfg)
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print("ANSWER Q7:", trainable)
lora_model.print_trainable_parameters()   # cross-check

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ANSWER Q7: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


In [9]:
from datasets import Dataset

def preprocess(ex):
    texts = [str(ex['prompt']) + " [SEP] " + str(ex[o]) for o in OPTS]
    enc = tok(texts, padding="max_length", truncation=True, max_length=128)
    return {"input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": ex["label"]}

ds100 = Dataset.from_pandas(train.head(100)[['prompt']+OPTS+['label']],
                            preserve_index=False).map(preprocess)
item0_ids = np.array(ds100[0]["input_ids"])
print("input_ids shape:", item0_ids.shape)     # (5, 128)
print("ANSWER Q8:", item0_ids.shape[0])        # 5 choices

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

input_ids shape: (5, 128)
ANSWER Q8: 5


In [10]:
from transformers import TrainingArguments, Trainer

def preprocess64(ex):
    texts = [str(ex['prompt']) + " [SEP] " + str(ex[o]) for o in OPTS]
    enc = tok(texts, padding="max_length", truncation=True, max_length=64)
    return {"input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": ex["label"]}

ds32 = Dataset.from_pandas(train.head(32)[['prompt']+OPTS+['label']],
                           preserve_index=False).map(preprocess64)

import torch as _t
def collate(features):
    return {"input_ids": _t.tensor([f["input_ids"] for f in features]),
            "attention_mask": _t.tensor([f["attention_mask"] for f in features]),
            "labels": _t.tensor([f["labels"] for f in features])}

tiny_args = TrainingArguments(output_dir="tiny_lora", max_steps=4,
                              per_device_train_batch_size=4,
                              gradient_accumulation_steps=1,
                              report_to=[], logging_steps=1, save_strategy="no")

tiny_trainer = Trainer(model=lora_model, args=tiny_args,
                       train_dataset=ds32, data_collator=collate)
result = tiny_trainer.train()
print("ANSWER Q9:", tiny_trainer.state.global_step)   # expect 4

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

c:\Users\SAGAR\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.544963
2,1.515999
3,1.623882
4,1.624588


ANSWER Q9: 4


In [11]:
lora_model.eval()
r0 = train.iloc[0]
texts0 = [str(r0['prompt']) + " [SEP] " + str(r0[o]) for o in OPTS]
enc0 = tok(texts0, padding="max_length", truncation=True,
           max_length=64, return_tensors="pt")
batch0 = {k: v.unsqueeze(0) for k, v in enc0.items()}

with torch.no_grad():
    logits = lora_model(**batch0).logits
probs = torch.softmax(logits, dim=1)[0]
print("all probs:", [round(p,4) for p in probs.tolist()])
print("ANSWER Q10:", round(probs[4].item(), 4))    # index 4 = Option E

all probs: [0.2053, 0.1856, 0.1919, 0.2081, 0.2091]
ANSWER Q10: 0.2091
